# 🇧🇯 Bénin Insights Challenge — Notebook EDA
## iSHEERO × DataCamp Donates 2026

---

**Équipe :** [Nom de votre équipe]  
**Rôle :** Data Analyste  
**Source des données :** GDELT — Global Database of Events, Language and Tone  
**Période analysée :** 12 derniers mois (avril 2025 – avril 2026)  
**Usage d'IA :** Ce notebook a été structuré avec l'assistance de Claude (Anthropic). Tous les commentaires analytiques et interprétations sont le travail de l'équipe.

---

### 🎯 Objectif
Extraire, analyser et visualiser les données GDELT relatives au Bénin pour produire des **insights utiles** pour un journaliste, un chercheur ou un décideur public.

### 📋 Structure du notebook
1. Chargement & nettoyage des données
2. **VIZ 1** — Évolution mensuelle du volume médiatique (Q1)
3. **VIZ 2** — Ton des médias par type d'événement (Q2)
4. **VIZ 3** — Top 15 pays citant le Bénin (Q3)
5. **VIZ 4** — Carte géographique des événements (Q4)
6. **VIZ 5** — Évolution du Score de Goldstein (Q5)
7. Synthèse des 5 insights clés

## ⚙️ Étape 0 — Installation des bibliothèques
Exécute cette cellule UNE SEULE FOIS au début.

In [ ]:
# Installation des bibliothèques nécessaires
# Lance cette cellule avec Shift+Entrée une seule fois
!pip install pandas plotly folium openpyxl --quiet

## 📂 Étape 1 — Chargement & Nettoyage des données

**⚠️ ACTION REQUISE :** Remplace `'benin_gdelt.csv'` par le nom exact du fichier que te donne le Data Engineer.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# ⚠️  MODIFIE ICI : mets le vrai nom de ton fichier CSV
# ============================================================
NOM_FICHIER = 'benin_gdelt.csv'

# Chargement
df = pd.read_csv(NOM_FICHIER, low_memory=False)

print(f'✅ Données chargées avec succès !')
print(f'📊 Nombre de lignes    : {len(df):,}')
print(f'📋 Nombre de colonnes  : {len(df.columns)}')
print(f'\n🔍 Aperçu des colonnes disponibles :')
print(df.columns.tolist())

In [ ]:
# ============================================================
# NETTOYAGE DES DONNÉES
# ============================================================

# 1. Conversion de la date (format GDELT : YYYYMMDD)
df['date'] = pd.to_datetime(df['SQLDATE'].astype(str), format='%Y%m%d', errors='coerce')

# 2. Création de colonnes utiles
df['mois']    = df['date'].dt.to_period('M').astype(str)   # ex: '2025-06'
df['annee']   = df['date'].dt.year
df['trimestre'] = df['date'].dt.to_period('Q').astype(str)

# 3. Suppression des lignes sans date valide
avant = len(df)
df = df.dropna(subset=['date'])
apres = len(df)
print(f'🗑️  Lignes supprimées (dates invalides) : {avant - apres}')

# 4. Aperçu statistique rapide
print(f'\n📅 Période couverte : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'\n📊 Statistiques clés :')
print(df[['GoldsteinScale','AvgTone','NumArticles']].describe().round(2))

---
## 📈 VIZ 1 — Évolution mensuelle du volume médiatique
### Question : Comment le volume de couverture médiatique internationale sur le Bénin a-t-il évolué ? Y a-t-il des pics significatifs ?

In [ ]:
# ============================================================
# VIZ 1 — Lineplot : Évolution mensuelle
# ============================================================

# Compter les événements par mois
df_monthly = df.groupby('mois').size().reset_index(name='nombre_evenements')
df_monthly = df_monthly.sort_values('mois')

# Calculer la moyenne pour la ligne de référence
moyenne = df_monthly['nombre_evenements'].mean()

# Identifier le pic maximum
idx_max = df_monthly['nombre_evenements'].idxmax()
mois_pic = df_monthly.loc[idx_max, 'mois']
val_pic  = df_monthly.loc[idx_max, 'nombre_evenements']

# Créer le graphique
fig1 = go.Figure()

# Zone remplie sous la courbe
fig1.add_trace(go.Scatter(
    x=df_monthly['mois'],
    y=df_monthly['nombre_evenements'],
    fill='tozeroy',
    fillcolor='rgba(29, 158, 117, 0.15)',
    line=dict(color='#1D9E75', width=2.5),
    mode='lines+markers',
    marker=dict(size=7, color='#1D9E75'),
    name="Événements / mois",
    hovertemplate='<b>%{x}</b><br>%{y} événements<extra></extra>'
))

# Ligne de moyenne
fig1.add_hline(
    y=moyenne,
    line_dash='dash',
    line_color='orange',
    annotation_text=f'Moyenne : {moyenne:.0f} evt/mois',
    annotation_position='top right'
)

# Annoter le pic
fig1.add_annotation(
    x=mois_pic, y=val_pic,
    text=f'📍 Pic : {val_pic} événements',
    showarrow=True, arrowhead=2,
    bgcolor='#E8F5F0', bordercolor='#1D9E75',
    font=dict(color='#0F6E56')
)

fig1.update_layout(
    title=dict(
        text='📈 Évolution mensuelle de la couverture médiatique du Bénin<br><sub>Source : GDELT — Bénin Insights Challenge 2026</sub>',
        font=dict(size=16)
    ),
    xaxis_title='Mois',
    yaxis_title="Nombre d'événements",
    template='plotly_white',
    xaxis_tickangle=-45,
    height=480,
    hovermode='x unified'
)

fig1.show()

### 💬 Commentaire analytique — VIZ 1

> **Ce que montre ce graphique :**  
> Le volume de couverture médiatique internationale sur le Bénin n'est pas uniforme sur la période. On observe des **pics significatifs** en certains mois qui dépassent nettement la moyenne.
>
> **À compléter quand tu verras les vraies données :**  
> *"Le pic le plus important s'observe en [MOIS/ANNÉE] avec [X] événements recensés. Cela coïncide probablement avec [événement politique/social au Bénin]. La tendance générale est [croissante / stable / décroissante] sur la période analysée, ce qui suggère que [interprétation]."*
>
> **Insight pour le jury :**  
> *"L'attention médiatique internationale sur le Bénin est cyclique et réactive à des événements précis — elle n'est pas continue."*

---
## 🎭 VIZ 2 — Ton des médias par type d'événement
### Question : Les médias internationaux parlent-ils du Bénin de façon positive ou négative selon les types d'événements couverts ?

In [ ]:
# ============================================================
# VIZ 2 — Boxplot : Ton moyen (AvgTone) par type d'événement
# ============================================================

# Dictionnaire de traduction des codes CAMEO → labels lisibles
cameo_labels = {
    '01': 'Déclarations publiques',
    '02': 'Appels / Demandes',
    '03': 'Expressions d\'intentions',
    '04': 'Consultations',
    '05': 'Engagement diplomatique',
    '06': 'Coopération matérielle',
    '07': 'Aide humanitaire',
    '08': 'Yield / Concessions',
    '09': 'Enquêtes judiciaires',
    '10': 'Demandes de changement',
    '11': 'Rejets / Refus',
    '12': 'Accusations',
    '13': 'Protestations',
    '14': 'Manifestations',
    '15': 'Menaces',
    '16': 'Sanctions / Embargo',
    '17': 'Arrestations',
    '18': 'Violences verbales',
    '19': 'Attaques armées',
    '20': 'Violences de masse'
}

# Top 8 types d'événements les plus fréquents
top_events = df['EventRootCode'].astype(str).str.zfill(2).value_counts().head(8).index.tolist()
df_viz2 = df[df['EventRootCode'].astype(str).str.zfill(2).isin(top_events)].copy()
df_viz2['EventRootCode_str'] = df_viz2['EventRootCode'].astype(str).str.zfill(2)
df_viz2['Type_evenement'] = df_viz2['EventRootCode_str'].map(cameo_labels).fillna('Code ' + df_viz2['EventRootCode_str'])

# Ordre par médiane du ton (du plus négatif au plus positif)
ordre = df_viz2.groupby('Type_evenement')['AvgTone'].median().sort_values().index.tolist()

# Couleurs : rouge = négatif, vert = positif
medians = df_viz2.groupby('Type_evenement')['AvgTone'].median()
colors = ['#D32F2F' if medians[t] < -1 else '#F57C00' if medians[t] < 0 else '#388E3C' for t in ordre]

fig2 = go.Figure()

for i, event_type in enumerate(ordre):
    subset = df_viz2[df_viz2['Type_evenement'] == event_type]['AvgTone']
    fig2.add_trace(go.Box(
        y=subset,
        name=event_type,
        marker_color=colors[i],
        boxmean=True,
        hovertemplate=f'<b>{event_type}</b><br>Ton : %{{y:.2f}}<extra></extra>'
    ))

# Ligne de neutralité à 0
fig2.add_hline(
    y=0,
    line_dash='dash',
    line_color='gray',
    line_width=1.5,
    annotation_text='Neutralité (0)',
    annotation_position='top right'
)

fig2.update_layout(
    title=dict(
        text='🎭 Ton des médias internationaux selon le type d\'événement au Bénin<br><sub>Rouge = couverture négative · Vert = couverture positive · Source : GDELT 2026</sub>',
        font=dict(size=16)
    ),
    yaxis_title='Ton moyen (AvgTone) — négatif < 0 < positif',
    xaxis_title="Type d'événement",
    showlegend=False,
    template='plotly_white',
    height=500,
    xaxis_tickangle=-30
)

fig2.show()

### 💬 Commentaire analytique — VIZ 2

> **Ce que montre ce graphique :**  
> Le ton médiatique (AvgTone) mesure l'état d'esprit des articles : en dessous de 0 = couverture négative, au-dessus = positive. Ce boxplot montre la distribution du ton pour chaque type d'événement.
>
> **À compléter quand tu verras les vraies données :**  
> *"Les événements de type [type X — ex: Attaques armées] reçoivent systématiquement une couverture très négative (médiane ≈ [valeur]). En revanche, les [type Y — ex: Coopération diplomatique] sont couverts positivement. Cela révèle que les médias associent principalement le Bénin à [type d'image]."*
>
> **Insight pour le jury :**  
> *"La perception internationale du Bénin est fortement influencée par [catégorie dominante]. Les décideurs publics devraient en tenir compte dans leur stratégie de communication."*

---
## 🌍 VIZ 3 — Top 15 pays qui citent le Bénin
### Question : Quels pays s'intéressent le plus au Bénin ? L'attention est-elle concentrée ou diversifiée ?

In [ ]:
# ============================================================
# VIZ 3 — Barplot horizontal : Top 15 pays sources
# ============================================================

# Dictionnaire codes pays → noms complets (principaux)
pays_noms = {
    'US': '🇺🇸 États-Unis', 'FR': '🇫🇷 France', 'GB': '🇬🇧 Royaume-Uni',
    'NG': '🇳🇬 Nigeria', 'SN': '🇸🇳 Sénégal', 'CI': '🇨🇮 Côte d\'Ivoire',
    'GH': '🇬🇭 Ghana', 'TG': '🇹🇬 Togo', 'NE': '🇳🇪 Niger',
    'CM': '🇨🇲 Cameroun', 'DE': '🇩🇪 Allemagne', 'CN': '🇨🇳 Chine',
    'RU': '🇷🇺 Russie', 'ML': '🇲🇱 Mali', 'BF': '🇧🇫 Burkina Faso',
    'MA': '🇲🇦 Maroc', 'ZA': '🇿🇦 Afrique du Sud', 'ET': '🇪🇹 Éthiopie',
    'IN': '🇮🇳 Inde', 'CA': '🇨🇦 Canada', 'BE': '🇧🇪 Belgique',
    'JP': '🇯🇵 Japon', 'BR': '🇧🇷 Brésil', 'AU': '🇦🇺 Australie',
    'BJ': '🇧🇯 Bénin (local)', 'GN': '🇬🇳 Guinée', 'BU': '🇧🇮 Burundi'
}

# Compter par pays source (Actor1CountryCode)
top_pays = df['Actor1CountryCode'].value_counts().head(15).reset_index()
top_pays.columns = ['code_pays', 'nombre_evenements']
top_pays['Pays'] = top_pays['code_pays'].map(pays_noms).fillna('🌐 ' + top_pays['code_pays'])
top_pays = top_pays.sort_values('nombre_evenements', ascending=True)

# Coloration : pays africains vs reste du monde
pays_afrique = ['NG', 'SN', 'CI', 'GH', 'TG', 'NE', 'CM', 'ML', 'BF', 'MA', 'ZA', 'ET', 'BJ', 'GN', 'BU']
top_pays['couleur'] = top_pays['code_pays'].apply(
    lambda x: '#1D9E75' if x in pays_afrique else '#185FA5'
)

fig3 = go.Figure(go.Bar(
    x=top_pays['nombre_evenements'],
    y=top_pays['Pays'],
    orientation='h',
    marker_color=top_pays['couleur'],
    text=top_pays['nombre_evenements'],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>%{x} événements<extra></extra>'
))

# Légende manuelle
fig3.add_trace(go.Bar(x=[None], y=[None], marker_color='#1D9E75', name='Pays africains', showlegend=True))
fig3.add_trace(go.Bar(x=[None], y=[None], marker_color='#185FA5', name='Reste du monde', showlegend=True))

fig3.update_layout(
    title=dict(
        text='🌍 Top 15 pays qui citent le Bénin dans les médias mondiaux<br><sub>Vert = pays africains · Bleu = reste du monde · Source : GDELT 2026</sub>',
        font=dict(size=16)
    ),
    xaxis_title="Nombre d'événements",
    yaxis_title='Pays',
    template='plotly_white',
    height=520,
    barmode='overlay'
)

fig3.show()

### 💬 Commentaire analytique — VIZ 3

> **Ce que montre ce graphique :**  
> Ce barplot révèle **d'où vient l'attention** que le monde porte au Bénin. La distinction vert/bleu montre si c'est la sous-région africaine ou les puissances mondiales qui parlent le plus du pays.
>
> **À compléter quand tu verras les vraies données :**  
> *"Les 3 pays qui s'intéressent le plus au Bénin sont [pays 1], [pays 2] et [pays 3]. [La France / Le Nigeria / Les États-Unis] dominent avec [X] événements, ce qui reflète [les liens historiques franco-béninois / la proximité géographique / les intérêts économiques]. L'attention est [concentrée sur 2-3 pays / diversifiée]."*
>
> **Insight pour le jury :**  
> *"La couverture du Bénin reste [très/peu] dominée par [pays], ce qui peut créer un biais dans la représentation internationale du pays."*

---
## 🗺️ VIZ 4 — Carte géographique des événements au Bénin
### Question : Les événements sont-ils concentrés dans certaines zones ou répartis sur tout le territoire ?

In [ ]:
# ============================================================
# VIZ 4 — Carte Scatter : Localisation géographique
# ============================================================

# Filtrer les lignes avec coordonnées valides AU BÉNIN
# Bénin : latitude 6.0 à 12.5 / longitude 0.5 à 3.9
df_geo = df.dropna(subset=['ActionGeo_Lat', 'ActionGeo_Long']).copy()
df_geo = df_geo[
    (df_geo['ActionGeo_Lat'].between(6.0, 12.5)) &
    (df_geo['ActionGeo_Long'].between(0.5, 3.9))
]

print(f'📍 Événements géolocalisés au Bénin : {len(df_geo):,}')
print(f'   Soit {len(df_geo)/len(df)*100:.1f}% du total')

# Regrouper par coordonnées et nom de lieu
df_grouped = df_geo.groupby(
    ['ActionGeo_Lat', 'ActionGeo_Long', 'ActionGeo_FullName']
).agg(
    nombre_evenements=('ActionGeo_Lat', 'count'),
    ton_moyen=('AvgTone', 'mean'),
    goldstein_moyen=('GoldsteinScale', 'mean')
).reset_index()

# Nettoyer le nom du lieu
df_grouped['Lieu'] = df_grouped['ActionGeo_FullName'].str.split(',').str[0]

fig4 = px.scatter_mapbox(
    df_grouped,
    lat='ActionGeo_Lat',
    lon='ActionGeo_Long',
    size='nombre_evenements',
    color='ton_moyen',
    hover_name='Lieu',
    hover_data={
        'nombre_evenements': True,
        'ton_moyen': ':.2f',
        'goldstein_moyen': ':.2f',
        'ActionGeo_Lat': False,
        'ActionGeo_Long': False
    },
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    size_max=45,
    zoom=6.2,
    center={'lat': 9.3, 'lon': 2.3},
    mapbox_style='carto-positron',
    title='🗺️ Carte des événements médiatiques géolocalisés au Bénin<br><sub>Taille = nombre d\'événements · Couleur = ton (rouge=négatif, vert=positif) · Source : GDELT 2026</sub>',
    height=600
)

fig4.update_layout(
    coloraxis_colorbar=dict(
        title='Ton moyen',
        tickvals=[-5, -2.5, 0, 2.5, 5],
        ticktext=['Très négatif', 'Négatif', 'Neutre', 'Positif', 'Très positif']
    ),
    margin=dict(l=0, r=0, t=80, b=0)
)

fig4.show()

### 💬 Commentaire analytique — VIZ 4

> **Ce que montre ce graphique :**  
> La taille de chaque bulle représente le nombre d'événements dans cette zone. La couleur indique si la couverture est positive (vert) ou négative (rouge). C'est la visualisation à **plus fort impact visuel** auprès du jury.
>
> **À compléter quand tu verras les vraies données :**  
> *"La grande majorité des événements médiatisés se concentre à [ville — probablement Cotonou]. Le nord du pays (départements de l'Alibori, Borgou) apparaît très peu dans la couverture internationale, ce qui révèle un biais géographique majeur. Les zones rouges (couverture négative) se situent principalement à [lieu], probablement liées à [événement]."*
>
> **Insight pour le jury :**  
> *"Les médias mondiaux réduisent le Bénin à [ville/région]. Le reste du territoire est quasi invisible internationalement."*

---
## ⚖️ VIZ 5 — Évolution du Score de Goldstein
### Question : La situation globale au Bénin s'améliore-t-elle ou se dégrade-t-elle selon le Score de Goldstein ?

In [ ]:
# ============================================================
# VIZ 5 — Lineplot lissé : Score de Goldstein dans le temps
# ============================================================

# Score de Goldstein moyen par mois
df_gold = df.groupby('mois').agg(
    score_goldstein=('GoldsteinScale', 'mean'),
    nombre_evt=('GoldsteinScale', 'count')
).reset_index().sort_values('mois')

# Moyenne mobile sur 3 mois pour lisser
df_gold['score_lisse'] = df_gold['score_goldstein'].rolling(window=3, center=True, min_periods=1).mean()

# Score global moyen sur toute la période
score_global = df_gold['score_goldstein'].mean()

# Créer le graphique
fig5 = go.Figure()

# Barres de fond (score brut mensuel)
fig5.add_trace(go.Bar(
    x=df_gold['mois'],
    y=df_gold['score_goldstein'],
    name='Score mensuel brut',
    marker_color=[
        'rgba(211, 47, 47, 0.4)' if v < 0 else 'rgba(56, 142, 60, 0.4)'
        for v in df_gold['score_goldstein']
    ],
    hovertemplate='<b>%{x}</b><br>Score brut : %{y:.2f}<extra></extra>'
))

# Courbe lissée principale
fig5.add_trace(go.Scatter(
    x=df_gold['mois'],
    y=df_gold['score_lisse'],
    mode='lines+markers',
    line=dict(color='#534AB7', width=3),
    marker=dict(size=8, color='#534AB7'),
    name='Tendance (moy. mobile 3 mois)',
    hovertemplate='<b>%{x}</b><br>Score lissé : %{y:.2f}<extra></extra>'
))

# Ligne de neutralité à 0
fig5.add_hline(
    y=0,
    line_dash='dash',
    line_color='black',
    line_width=1,
    annotation_text='Neutralité (0)',
    annotation_position='top left'
)

# Ligne du score global moyen
fig5.add_hline(
    y=score_global,
    line_dash='dot',
    line_color='orange',
    line_width=2,
    annotation_text=f'Moyenne globale : {score_global:.2f}',
    annotation_position='bottom right'
)

fig5.update_layout(
    title=dict(
        text='⚖️ Évolution du Score de Goldstein au Bénin<br><sub>Rouge = instabilité/conflit · Vert = coopération/stabilité · Échelle : -10 à +10 · Source : GDELT 2026</sub>',
        font=dict(size=16)
    ),
    xaxis_title='Mois',
    yaxis_title='Score de Goldstein moyen (-10 = conflictuel, +10 = coopératif)',
    template='plotly_white',
    xaxis_tickangle=-45,
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)

fig5.show()

print(f'\n📊 Score de Goldstein moyen sur la période : {score_global:.2f}')
print(f'   → Interprétation : {"Contexte globalement COOPÉRATIF ✅" if score_global > 0 else "Contexte globalement CONFLICTUEL ⚠️"}')

### 💬 Commentaire analytique — VIZ 5

> **Ce que montre ce graphique :**  
> Le Score de Goldstein mesure la stabilité globale : +10 = coopération totale, -10 = conflit total. Les barres rouges indiquent des mois où les événements couverts étaient plutôt conflictuels ; les barres vertes, des périodes plus stables.
>
> **À compléter quand tu verras les vraies données :**  
> *"Sur la période analysée, le score moyen de Goldstein pour le Bénin est de [valeur]. Les mois de [mois] ont enregistré les scores les plus négatifs, coïncidant avec [événements : tensions électorales, insécurité au nord...]. On observe [une amélioration / une dégradation / une stabilité] depuis [période]."*
>
> **Insight pour le jury :**  
> *"Malgré [événements négatifs], la trajectoire du Bénin semble [s'améliorer / se stabiliser / se dégrader] — un signal [encourageant / préoccupant] pour les décideurs."*

---
## 🏆 SYNTHÈSE — Les 5 Insights Clés
*Section à compléter après analyse des vraies données — à inclure dans le résumé d'une page et la vidéo pitch*

In [ ]:
# ============================================================
# TABLEAU RÉCAPITULATIF DES 5 INSIGHTS
# ============================================================

# Calcul des métriques clés automatiques
print('=' * 65)
print('  🇧🇯 BÉNIN INSIGHTS CHALLENGE — SYNTHÈSE ANALYTIQUE 2026')
print('=' * 65)

print(f'\n📊 DONNÉES ANALYSÉES')
print(f'   Total événements GDELT : {len(df):,}')
print(f'   Période               : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'   Pays sources distincts : {df["Actor1CountryCode"].nunique()}')

print(f'\n🔍 INSIGHT 1 — VOLUME MÉDIATIQUE')
print(f'   Mois le plus couvert  : {df_monthly.loc[df_monthly["nombre_evenements"].idxmax(), "mois"]}')
print(f'   Pic d\'événements     : {df_monthly["nombre_evenements"].max():,}')
print(f'   → [Compléter : pourquoi ce pic ?]')

print(f'\n🔍 INSIGHT 2 — SENTIMENT MÉDIATIQUE')
print(f'   Ton moyen global      : {df["AvgTone"].mean():.2f}')
print(f'   → {"Couverture globalement NÉGATIVE ⚠️" if df["AvgTone"].mean() < 0 else "Couverture globalement POSITIVE ✅"}')

print(f'\n🔍 INSIGHT 3 — ATTENTION INTERNATIONALE')
top3 = df['Actor1CountryCode'].value_counts().head(3)
for code, count in top3.items():
    nom = pays_noms.get(code, code)
    print(f'   {nom} : {count:,} événements')

print(f'\n🔍 INSIGHT 4 — GÉOGRAPHIE')
print(f'   Événements géolocalisés au Bénin : {len(df_geo):,}')
if len(df_grouped) > 0:
    top_lieu = df_grouped.nlargest(1, 'nombre_evenements').iloc[0]
    print(f'   Zone la plus couverte : {top_lieu["Lieu"]} ({top_lieu["nombre_evenements"]:,} evt)')

print(f'\n🔍 INSIGHT 5 — STABILITÉ (GOLDSTEIN)')
print(f'   Score moyen           : {score_global:.2f} / 10')
print(f'   → {"Période globalement STABLE ✅" if score_global > 0 else "Période globalement INSTABLE ⚠️"}')

print(f'\n' + '=' * 65)
print('  ✅ Notebook EDA complet — Prêt pour soumission GitHub')
print('=' * 65)

---

## ✅ Checklist avant soumission (Mardi 5 mai 23h59)

- [ ] Le notebook tourne de A à Z sans erreur (Run All)
- [ ] Les 5 commentaires analytiques sont remplis avec les vraies observations
- [ ] Le notebook est uploadé dans le dossier `/notebooks` du repo GitHub
- [ ] Le fichier `requirements.txt` est à jour
- [ ] L'usage d'IA est mentionné dans le README
- [ ] Le dashboard Streamlit est déployé et accessible en ligne
- [ ] La vidéo pitch (3 min max) est uploadée sur YouTube ou Google Drive
- [ ] Le résumé d'une page (5 insights) est prêt en PDF

---
*Bénin Insights Challenge 2026 — iSHEERO × DataCamp Donates*  
*Rôle : Data Analyste | Usage IA : Claude (Anthropic)*